<a href="https://colab.research.google.com/github/kartik815/Amazon-ML-Challenge-2026/blob/main/notebooks/04_Blocking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
from google.colab import drive

drive.mount('/content/drive')

import os
import gc
import pandas as pd

DRIVE_ROOT = "/content/drive/MyDrive/Amazon ML Challenge 2026"

CLEANED_DATA_ROOT = os.path.join(
    DRIVE_ROOT,
    "03_Experiments",
    "Cleaned_Data"
)

TRAIN_ROOT = os.path.join(
    DRIVE_ROOT,
    "01_Dataset",
    "6ab10eb3b23ba_student_resource",
    "student_resource",
    "dataset",
    "train"
)

TEST_ROOT = os.path.join(
    DRIVE_ROOT,
    "01_Dataset",
    "6ab10eb3b23ba_student_resource",
    "student_resource",
    "dataset",
    "test"
)

BLOCKING_OUTPUT_ROOT = os.path.join(
    DRIVE_ROOT,
    "05_Outputs",
    "Candidate_Pairs"
)

os.makedirs(BLOCKING_OUTPUT_ROOT, exist_ok=True)

print("Cleaned data:")
print(CLEANED_DATA_ROOT)

print("\nBlocking output:")
print(BLOCKING_OUTPUT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cleaned data:
/content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data

Blocking output:
/content/drive/MyDrive/Amazon ML Challenge 2026/05_Outputs/Candidate_Pairs


In [23]:
CLEANED_FILES = [
    "train_s1_cleaned.tsv",
    "train_s2_cleaned.tsv",
    "train_s3_cleaned.tsv",
    "test_s1_cleaned.tsv",
    "test_s2_cleaned.tsv",
    "test_s3_cleaned.tsv"
]

print("Checking cleaned files...\n")

for filename in CLEANED_FILES:

    path = os.path.join(
        CLEANED_DATA_ROOT,
        filename
    )

    print(
        f"{filename:<25}",
        "EXISTS" if os.path.exists(path) else "MISSING"
    )

Checking cleaned files...

train_s1_cleaned.tsv      EXISTS
train_s2_cleaned.tsv      EXISTS
train_s3_cleaned.tsv      EXISTS
test_s1_cleaned.tsv       EXISTS
test_s2_cleaned.tsv       EXISTS
test_s3_cleaned.tsv       EXISTS


In [24]:
sample_path = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s1_cleaned.tsv"
)

sample = pd.read_csv(
    sample_path,
    sep="\t",
    nrows=5
)

print("Columns:")
print(sample.columns.tolist())

print("\nSample:")
display(sample)

Columns:
['entity_id', 'business_name', 'business_address', 'country', 'name_norm', 'address_norm', 'country_norm', 'name_missing', 'address_missing', 'name_token_count', 'address_token_count', 'name_core']

Sample:


,entity_id,business_name,business_address,country,name_norm,address_norm,country_norm,name_missing,address_missing,name_token_count,address_token_count,name_core
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,orelee s barbershop,1795 westchester dr high point nc,us,0,0,3,6,orelee s barbershop
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,prime money,17560 ellis rd tahlequah ok,us,0,0,2,5,prime money
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,b retail inc,1712 montebello ave phoenix az,us,0,0,3,5,b retail
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,christ chapel,2100 cameron dr unit apt g dundalk md,us,0,0,2,8,christ chapel
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,prabhav business center,797 lake town block a kolkata howrah west bengal,india,0,0,3,9,prabhav business center


Blocker 1 (country_norm + name_core)

In [25]:
import pandas as pd
from collections import Counter

S2_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s2_cleaned.tsv"
)

S3_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s3_cleaned.tsv"
)

CHUNK_SIZE = 200_000

def get_block_sizes(path):
  counter = Counter()

  for chunk in pd.read_csv(
      path,
      sep="\t",
      chunksize=CHUNK_SIZE,
      dtype={"country_norm": "string", "name_norm": "string"}):
      chunk = chunk[
          chunk["name_core"].notna() &
          (chunk["name_core"] != "")
      ]

      keys = zip(
          chunk["country_norm"],
          chunk["name_core"]
      )

      counter.update(keys)

      del chunk
      gc.collect()

  return counter

print("Building S2 block statistics...")
s2_blocks = get_block_sizes(S2_PATH)

print("Building S3 block statistics...")
s3_blocks = get_block_sizes(S3_PATH)

print("\nS2 unique blocks:", len(s2_blocks))
print("S3 unique blocks:", len(s3_blocks))


Building S2 block statistics...
Building S3 block statistics...

S2 unique blocks: 3597772
S3 unique blocks: 3850257


In [26]:
print("\nLargest S2 blocks:")
for key, count in s2_blocks.most_common(20):
    print(count, "->", key)

print("\nLargest S3 blocks:")
for key, count in s3_blocks.most_common(20):
    print(count, "->", key)


Largest S2 blocks:
647 -> ('us', 'meridian')
563 -> ('us', 'physical therapy')
550 -> ('us', 'primary care')
514 -> ('us', 'womens health')
507 -> ('us', 'internal medicine')
500 -> ('us', 'behavioral health')
494 -> ('us', 'urgent care')
491 -> ('us', 'pediatric dental')
482 -> ('us', 'pediatric dentistry')
408 -> ('us', 'ear nose throat')
398 -> ('us', 'foot ankle')
375 -> ('us', 'family center')
374 -> ('us', 'summit')
374 -> ('us', 'lynx')
366 -> ('us', 'anchor')
362 -> ('us', 'sapphire')
360 -> ('us', 'helios')
359 -> ('us', 'falcon')
356 -> ('us', 'cedar')
355 -> ('us', 'earnosethroat com')

Largest S3 blocks:
584 -> ('us', 'meridian')
548 -> ('us', 'primary care')
538 -> ('us', 'physical therapy')
524 -> ('us', 'pediatric dental')
507 -> ('us', 'urgent care')
492 -> ('us', 'womens health')
490 -> ('us', 'pediatric dentistry')
458 -> ('us', 'behavioral health')
452 -> ('us', 'internal medicine')
426 -> ('us', 'summit')
418 -> ('us', 'cascade')
412 -> ('us', 'family center')
399 

In [27]:
from collections import defaultdict
import gc

def build_block_index(path):

    index = defaultdict(list)

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "country_norm", "name_core"],
        chunksize=200_000,
        dtype={
            "entity_id": "string",
            "country_norm": "string",
            "name_core": "string"
        }
    ):

        chunk = chunk[
            chunk["name_core"].notna() &
            (chunk["name_core"] != "")
        ]

        for country, name, entity_id in zip(
            chunk["country_norm"],
            chunk["name_core"],
            chunk["entity_id"]
        ):
            index[(country, name)].append(entity_id)

        del chunk
        gc.collect()

    return index

In [28]:
print("Building S2 index...")
s2_index = build_block_index(S2_PATH)
print("S2 index blocks:", len(s2_index))
gc.collect()
print("\nBuilding S3 index...")
s3_index = build_block_index(S3_PATH)
print("S3 index blocks:", len(s3_index))

Building S2 index...
S2 index blocks: 3597772

Building S3 index...
S3 index blocks: 3850257


In [29]:
GT_PATH = os.path.join(
    TRAIN_ROOT,
    "train_ground_truth.tsv"
)

S1_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s1_cleaned.tsv"
)

gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype={
        "source1_entity_id": "string",
        "matched_entity_ids": "string"
    }
)

s1 = pd.read_csv(
    S1_PATH,
    sep="\t",
    usecols=[
        "entity_id",
        "country_norm",
        "name_core"
    ],
    dtype={
        "entity_id": "string",
        "country_norm": "string",
        "name_core": "string"
    }
)

print("S1 rows:", len(s1))
print("Ground truth rows:", len(gt))

S1 rows: 2206821
Ground truth rows: 2206821


In [30]:
gt_lookup = {}

for row in gt.itertuples(index=False):
    s1_id = row.source1_entity_id
    matched = row.matched_entity_ids

    if pd.isna(matched) or matched == "":
        gt_lookup[s1_id] = set()
    else:
        gt_lookup[s1_id] = {
            x.strip()
            for x in str(matched).split(",")
            if x.strip()
        }

print("Ground truth lookup entries:", len(gt_lookup))

Ground truth lookup entries: 2206821


In [31]:
def calculate_block_recall(
    s1_df,
    gt_lookup,
    index,
    prefix,
    block_column
):
    total_matches = 0
    captured_matches = 0

    for row in s1_df.itertuples(index=False):

        s1_id = row.entity_id
        country = row.country_norm
        block_value = getattr(row, block_column)

        true_ids = {
            x for x in gt_lookup.get(s1_id, set())
            if x.startswith(prefix)
        }

        if not true_ids:
            continue

        total_matches += len(true_ids)

        if pd.isna(block_value) or block_value == "":
            candidates = []
        else:
            candidates = index.get(
                (country, block_value),
                []
            )

        captured_matches += len(
            true_ids.intersection(candidates)
        )

    recall = (
        captured_matches / total_matches
        if total_matches
        else 0
    )

    return total_matches, captured_matches, recall

In [35]:
s2_total, s2_captured, s2_recall = calculate_block_recall(
    s1_blocking,
    gt_lookup,
    s2_name_index,
    "S2-",
    "name_core"
)

print("S2")
print("Total true matches :", s2_total)
print("Captured matches   :", s2_captured)
print(f"Blocking recall    : {s2_recall:.4%}")

S2
Total true matches : 3693619
Captured matches   : 1447011
Blocking recall    : 39.1760%


In [36]:
s3_address_index = build_address_index(S3_PATH)

s3_addr_total, s3_addr_captured, s3_addr_recall = calculate_block_recall(
    s1_blocking,
    gt_lookup,
    s3_address_index,
    "S3-",
    "address_norm"
)

print("S3")
print("Total true matches :", s3_addr_total)
print("Captured matches   :", s3_addr_captured)
print(f"Address + country recall: {s3_addr_recall:.4%}")

S3
Total true matches : 3944746
Captured matches   : 170441
Address + country recall: 4.3207%


In [37]:
del s2_index
del s3_index
gc.collect()
print("Name indexes cleared from memory.")

Name indexes cleared from memory.


In [32]:
from collections import defaultdict
import gc
import pandas as pd

def build_address_index(path):
    index = defaultdict(list)

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "country_norm", "address_norm"],
        chunksize=200_000,
        dtype={
            "entity_id": "string",
            "country_norm": "string",
            "address_norm": "string"
        }
    ):
        chunk = chunk[
            chunk["address_norm"].notna() &
            (chunk["address_norm"] != "")
        ]

        for country, address, entity_id in zip(
            chunk["country_norm"],
            chunk["address_norm"],
            chunk["entity_id"]
        ):
            index[(country, address)].append(entity_id)

        del chunk
        gc.collect()

    return index

In [38]:
print("Building S2 address index...")
s2_address_index = build_address_index(S2_PATH)
print("S2 address blocks:", len(s2_address_index))

Building S2 address index...
S2 address blocks: 4090600


In [39]:
s2_addr_total, s2_addr_captured, s2_addr_recall = calculate_block_recall(
    s1_blocking,
    gt_lookup,
    s2_address_index,
    "S2-",
    "address_norm"
)

print("S2")
print("Total true matches :", s2_addr_total)
print("Captured matches   :", s2_addr_captured)
print(f"Address + country recall: {s2_addr_recall:.4%}")

S2
Total true matches : 3693619
Captured matches   : 719297
Address + country recall: 19.4740%


In [40]:
del s2_address_index
gc.collect()
print("S2 address index cleared.")

S2 address index cleared.


In [41]:
print("Building S3 address index...")
s3_address_index = build_address_index(S3_PATH)
print("S3 address blocks:", len(s3_address_index))

Building S3 address index...
S3 address blocks: 4437787


In [42]:
s3_address_index = build_address_index(S3_PATH)

s3_addr_total, s3_addr_captured, s3_addr_recall = calculate_block_recall(
    s1_blocking,
    gt_lookup,
    s3_address_index,
    "S3-",
    "address_norm"
)

print("S3")
print("Total true matches :", s3_addr_total)
print("Captured matches   :", s3_addr_captured)
print(f"Address + country recall: {s3_addr_recall:.4%}")

S3
Total true matches : 3944746
Captured matches   : 170441
Address + country recall: 4.3207%


In [43]:
del s3_address_index
gc.collect()

33

In [44]:
# Show some S1 entities that have known S2 matches
# and compare their normalized name/address fields.

sample_s1_ids = []

for s1_id, matches in gt_lookup.items():
    s2_matches = [x for x in matches if x.startswith("S2-")]
    if s2_matches:
        sample_s1_ids.append((s1_id, s2_matches[:3]))

    if len(sample_s1_ids) >= 10:
        break

sample_s1_ids

[('S1-965667', ['S2-681193310', 'S2-743505751']),
 ('S1-55344266', ['S2-197070651', 'S2-249013014']),
 ('S1-343815751', ['S2-790675320', 'S2-479876582']),
 ('S1-656753428', ['S2-24659151', 'S2-153058913']),
 ('S1-102811957', ['S2-625774905', 'S2-553508714', 'S2-478959098']),
 ('S1-18727616', ['S2-755677256']),
 ('S1-318373630', ['S2-660036492']),
 ('S1-29845983', ['S2-648035184']),
 ('S1-789009573', ['S2-383871912']),
 ('S1-730934468', ['S2-356983532'])]

In [45]:
sample_ids = [x[0] for x in sample_s1_ids]
sample_s2_ids = [
    match
    for _, matches in sample_s1_ids
    for match in matches
]

s1_sample = pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype="string"
)

s2_sample = pd.read_csv(
    S2_PATH,
    sep="\t",
    dtype="string"
)

s1_sample = s1_sample[
    s1_sample["entity_id"].isin(sample_ids)
]

s2_sample = s2_sample[
    s2_sample["entity_id"].isin(sample_s2_ids)
]

print("S1 samples:")
display(
    s1_sample[
        ["entity_id", "business_name", "business_address",
         "name_core", "address_norm"]
    ]
)

print("\nS2 matching samples:")
display(
    s2_sample[
        ["entity_id", "business_name", "business_address",
         "name_core", "address_norm"]
    ]
)

S1 samples:


,entity_id,business_name,business_address,name_core,address_norm
539414,S1-102811957,Payne Enterprises,"3315 Fremont Street, Peoria, IL",payne enterprises,3315 fremont st peoria il
547338,S1-343815751,Dahlia Power Reliable Scientific LLC,"630 45th Terrace, Kansas City, MO",dahlia power reliable scientific,630 45th terrace kansas city mo
603899,S1-318373630,Red Ventures Private Limited,"Rajasthan, Jaipur, Banipark, Gokul Apartment, ...",red ventures,rajasthan jaipur banipark gokul apt e 3a kanti...
1043869,S1-730934468,Orellana Investments LLC,"728 A Quail Avenue, Fl Ground Floor, Geneva, IA",orellana investments,728 a quail ave fl ground floor geneva ia
1072115,S1-656753428,Ss Food Private Limited,"Af-684, Nandgram Near Mother India Public Scho...",ss food,af 684 nandgram near mother india public schoo...
1232493,S1-29845983,Hendricks and Flowers Inc,"33 Sleepy Hollow Drive, Danbury, CT",hendricks and flowers,33 sleepy hollow dr danbury ct
1265942,S1-789009573,Hotel Enterprises Limited,"Wz-187C Shop No.13, 14 Kh. No.47 S/F. Vikaspur...",hotel enterprises,wz 187c shop no 13 14 kh no 47 s f vikaspuri b...
1286323,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",maure williams colombier,85 wayne ave ticonderoga ny
1898165,S1-55344266,Raj Investments LLP,"6(29), C.I.T. Colony, 2Nd Main Road Mylapore, ...",raj investments,6 29 c i t colony 2nd main rd mylapore chennai...
2117972,S1-18727616,Lumay Boral,"1056 Belden Avenue, Akron, OH",lumay boral,1056 belden ave akron oh



S2 matching samples:


,entity_id,business_name,business_address,name_core,address_norm
191919,S2-648035184,Hendricks and Flowers Inc,"CT, SLEEPY HOLLOW DRIVE, DANBURY",hendricks and flowers,ct sleepy hollow dr danbury
223808,S2-755677256,Lumay Boral Inc.,"1056-1060 BELDEN AVE, PO BOX 8807, AKRON, OH",lumay boral,1056 1060 belden ave po box 8807 akron oh
385243,S2-625774905,PAYNE-ENRTPRMISES,"3315 FREMONT SAINT, PEORIA, IL",payne enrtprmises,3315 fremont saint peoria il
488130,S2-383871912,होटल एंटरप्राइजेज लिमिटेड,"WZ-187C SHOP NO.13, DELHI, WEST DELHI, Delhi",ह टल ए टरप र इज ज ल म ट ड,wz 187c shop no 13 delhi west delhi delhi
731099,S2-478959098,Payne Énterprises,"3315 FREMONT ST, PEORIA, IL",payne énterprises,3315 fremont st peoria il
809231,S2-660036492,रेड वेंचर्स प्राइवेट लिमिटेड,"G-1, BANIPARK, JAIPUR, Rajasthan",र ड व चर स प र इव ट ल म ट ड,g 1 banipark jaipur rajasthan
1376739,S2-681193310,Maure Wilblims Colombier Inc,<NA>,maure wilblims colombier,<NA>
1776076,S2-553508714,Payne Enterpires,"3315 FREMONT ST, PEORIA, IL",payne enterpires,3315 fremont st peoria il
1886844,S2-197070651,Raj Investments LLP,"6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, ...",raj investments,6 29 c i t colony 2nd main rd mylapore chennai...
1901972,S2-479876582,Dahlia Power Reliable Scientific,"45ND TERRACE, null, KANSAS CITY, MO",dahlia power reliable scientific,45nd terrace null kansas city mo


In [46]:
from collections import Counter
import pandas as pd
import gc

token_counts = Counter()

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):
    for name in chunk["name_core"].dropna():
        tokens = set(name.split())

        for token in tokens:
            if token:
                token_counts[token] += 1

    del chunk
    gc.collect()

print("Unique name tokens:", len(token_counts))

print("\nMost common tokens:")
for token, count in token_counts.most_common(30):
    print(f"{token:30} {count:,}")

Unique name tokens: 810506

Most common tokens:
ल                              237,102
ट                              232,862
र                              226,825
ड                              208,515
प                              206,868
म                              206,488
com                            201,464
center                         193,696
ltd                            161,428
स                              160,325
इव                             157,955
partners                       152,694
pvt                            147,583
services                       145,239
group                          139,390
s                              138,303
llc                            121,171
and                            118,201
c                              113,281
india                          110,684
क                              105,891
holdings                       97,019
l                              93,771
care                           88,788
of                 

In [47]:
NAME_STOPWORDS = {
    "inc", "incorporated",
    "llc",
    "ltd", "limited",
    "pvt", "private",
    "company", "co",
    "corp", "corporation",
    "group",
    "services",
    "partners",
    "holdings",
    "associates",
    "center",
    "india",
    "and",
    "of"
}

def meaningful_tokens(name):
    if pd.isna(name) or not str(name).strip():
        return []

    tokens = str(name).split()

    return [
        token for token in tokens
        if token not in NAME_STOPWORDS
        and len(token) >= 2
    ]

In [48]:
examples = [
    "payne enterprises",
    "payne enrtprmises",
    "maure williams colombier",
    "maure wilblims colombier",
    "dahlia power reliable scientific",
    "dahlia power reliable",
    "raj investments",
    "ss food",
    "orellana investments investments"
]

for name in examples:
    print(name, "->", meaningful_tokens(name))

payne enterprises -> ['payne', 'enterprises']
payne enrtprmises -> ['payne', 'enrtprmises']
maure williams colombier -> ['maure', 'williams', 'colombier']
maure wilblims colombier -> ['maure', 'wilblims', 'colombier']
dahlia power reliable scientific -> ['dahlia', 'power', 'reliable', 'scientific']
dahlia power reliable -> ['dahlia', 'power', 'reliable']
raj investments -> ['raj', 'investments']
ss food -> ['ss', 'food']
orellana investments investments -> ['orellana', 'investments', 'investments']


In [49]:
from collections import Counter
import gc
import pandas as pd

meaningful_token_counts = Counter()

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):
    for name in chunk["name_core"].dropna():
        for token in set(meaningful_tokens(name)):
            meaningful_token_counts[token] += 1

    del chunk
    gc.collect()

print("Unique meaningful tokens:", len(meaningful_token_counts))

print("\nMost common meaningful tokens:")
for token, count in meaningful_token_counts.most_common(30):
    print(f"{token:30} {count:,}")

Unique meaningful tokens: 810209

Most common meaningful tokens:
com                            201,464
इव                             157,955
care                           88,788
service                        55,261
health                         51,338
enterprises                    49,341
lp                             47,941
industries                     47,928
clinic                         46,246
ventures                       44,338
the                            42,897
solutions                      41,013
public                         36,965
pc                             35,772
global                         35,242
brothers                       35,234
trading                        35,098
pediatric                      32,974
medicine                       31,536
technologies                   30,230
exports                        29,692
यर                             29,249
dental                         26,003
dr                             25,806
shri                 

In [50]:
from collections import Counter

frequency_buckets = {
    "1": 0,
    "2-5": 0,
    "6-10": 0,
    "11-50": 0,
    "51-100": 0,
    "101-500": 0,
    "501-1000": 0,
    "1001-5000": 0,
    "5001-10000": 0,
    "10001-50000": 0,
    "50001+": 0
}

for token, count in meaningful_token_counts.items():

    if count == 1:
        frequency_buckets["1"] += 1
    elif count <= 5:
        frequency_buckets["2-5"] += 1
    elif count <= 10:
        frequency_buckets["6-10"] += 1
    elif count <= 50:
        frequency_buckets["11-50"] += 1
    elif count <= 100:
        frequency_buckets["51-100"] += 1
    elif count <= 500:
        frequency_buckets["101-500"] += 1
    elif count <= 1000:
        frequency_buckets["501-1000"] += 1
    elif count <= 5000:
        frequency_buckets["1001-5000"] += 1
    elif count <= 10000:
        frequency_buckets["5001-10000"] += 1
    elif count <= 50000:
        frequency_buckets["10001-50000"] += 1
    else:
        frequency_buckets["50001+"] += 1

for bucket, count in frequency_buckets.items():
    print(f"{bucket:>12}: {count:,}")

           1: 640,328
         2-5: 88,666
        6-10: 24,300
       11-50: 38,107
      51-100: 9,998
     101-500: 6,224
    501-1000: 1,273
   1001-5000: 911
  5001-10000: 161
 10001-50000: 236
      50001+: 5


In [51]:
from collections import defaultdict
import pandas as pd
import gc

MIN_TOKEN_FREQ = 2
MAX_TOKEN_FREQ = 500

# Keeps only tokens that occur in the useful frequency range
allowed_tokens = {
    token
    for token, count in meaningful_token_counts.items()
    if MIN_TOKEN_FREQ <= count <= MAX_TOKEN_FREQ
}

print("Allowed tokens:", len(allowed_tokens))

Allowed tokens: 167295


In [52]:
def build_token_index(path, allowed_tokens):
    index = defaultdict(list)

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "name_core"],
        chunksize=200_000,
        dtype={
            "entity_id": "string",
            "name_core": "string"
        }
    ):
        for entity_id, name in zip(
            chunk["entity_id"],
            chunk["name_core"]
        ):
            if pd.isna(name):
                continue

            tokens = set(meaningful_tokens(name))

            for token in tokens:
                if token in allowed_tokens:
                    index[token].append(entity_id)

        del chunk
        gc.collect()

    return index

In [53]:
print("Building S2 token index...")
s2_token_index = build_token_index(
    S2_PATH,
    allowed_tokens
)

print("S2 token blocks:", len(s2_token_index))

Building S2 token index...
S2 token blocks: 167295


In [54]:
def calculate_token_block_recall(
    s1_df,
    gt_lookup,
    token_index,
    prefix
):
    total_matches = 0
    captured_matches = 0

    for row in s1_df.itertuples(index=False):

        s1_id = row.entity_id
        name = row.name_core

        true_ids = {
            x for x in gt_lookup.get(s1_id, set())
            if x.startswith(prefix)
        }

        if not true_ids:
            continue

        total_matches += len(true_ids)

        if pd.isna(name):
            continue

        tokens = set(meaningful_tokens(name))

        candidate_set = set()

        for token in tokens:
            if token in allowed_tokens:
                candidate_set.update(token_index.get(token, []))

        captured_matches += len(
            true_ids.intersection(candidate_set)
        )

    recall = (
        captured_matches / total_matches
        if total_matches else 0
    )

    return total_matches, captured_matches, recall

In [55]:
s2_token_total, s2_token_captured, s2_token_recall = \
    calculate_token_block_recall(
        s1,
        gt_lookup,
        s2_token_index,
        "S2-")

print("S2 total true matches:", s2_token_total)
print("S2 captured matches:", s2_token_captured)
print(f"S2 token blocking recall: {s2_token_recall:.4%}")

S2 total true matches: 3693619
S2 captured matches: 1568457
S2 token blocking recall: 42.4640%


In [56]:
def token_candidate_stats(s1_df, token_index, sample_size=10000):
    candidate_counts = []

    for i, row in enumerate(s1_df.itertuples(index=False)):

        if i >= sample_size:
            break

        name = row.name_core

        if pd.isna(name):
            candidate_counts.append(0)
            continue

        tokens = set(meaningful_tokens(name))

        candidate_set = set()

        for token in tokens:
            if token in allowed_tokens:
                candidate_set.update(
                    token_index.get(token, [])
                )

        candidate_counts.append(len(candidate_set))

    s = pd.Series(candidate_counts)

    return {
        "sample_size": len(s),
        "mean": s.mean(),
        "median": s.median(),
        "p90": s.quantile(0.90),
        "p95": s.quantile(0.95),
        "p99": s.quantile(0.99),
        "max": s.max(),
        "zero_candidates": (s == 0).sum()
    }

In [57]:
s2_token_stats = token_candidate_stats(
    s1,
    s2_token_index,
    sample_size=10000
)

s2_token_stats

{'sample_size': 10000,
 'mean': np.float64(69.5515),
 'median': 0.0,
 'p90': np.float64(242.10000000000036),
 'p95': np.float64(357.0),
 'p99': np.float64(517.0),
 'max': 995,
 'zero_candidates': np.int64(5161)}

In [58]:
from collections import defaultdict
import pandas as pd
import gc

def build_name_index(path):
    index = defaultdict(list)

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "country_norm", "name_core"],
        chunksize=200_000,
        dtype={
            "entity_id": "string",
            "country_norm": "string",
            "name_core": "string"
        }
    ):
        chunk = chunk[
            chunk["name_core"].notna() &
            (chunk["name_core"] != "")
        ]

        for country, name, entity_id in zip(
            chunk["country_norm"],
            chunk["name_core"],
            chunk["entity_id"]
        ):
            index[(country, name)].append(entity_id)

        del chunk
        gc.collect()

    return index

In [34]:
print("Building S2 exact-name index...")

s2_name_index = build_name_index(S2_PATH)

print("S2 name blocks:", len(s2_name_index))

Building S2 exact-name index...
S2 name blocks: 3597772


In [59]:
def calculate_union_recall(
    s1_df,
    gt_lookup,
    name_index,
    token_index,
    allowed_tokens,
    prefix
):
    total_matches = 0
    captured_matches = 0

    for row in s1_df.itertuples(index=False):
        s1_id = row.entity_id
        country = row.country_norm
        name = row.name_core
        true_ids = {
            x for x in gt_lookup.get(s1_id, set())
            if x.startswith(prefix)
        }

        if not true_ids:
            continue

        total_matches += len(true_ids)
        candidate_set = set()

        if pd.notna(name):
            candidate_set.update(
                name_index.get((country, name), [])
            )

        if pd.notna(name):

            tokens = set(meaningful_tokens(name))
            for token in tokens:
                if token in allowed_tokens:
                    candidate_set.update(
                        token_index.get(token, [])
                    )

        captured_matches += len(
            true_ids.intersection(candidate_set)
        )

    recall = (
        captured_matches / total_matches
        if total_matches else 0
    )

    return total_matches, captured_matches, recall

In [60]:
s2_union_total, s2_union_captured, s2_union_recall = \
    calculate_union_recall(
        s1,
        gt_lookup,
        s2_name_index,
        s2_token_index,
        allowed_tokens,
        "S2-"
    )

print("S2 total true matches:", s2_union_total)
print("S2 captured matches:", s2_union_captured)
print(f"S2 union blocking recall: {s2_union_recall:.4%}")

S2 total true matches: 3693619
S2 captured matches: 2255373
S2 union blocking recall: 61.0613%


In [61]:
address_token_counts = Counter()
for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["address_norm"],
    chunksize=200_000,
    dtype={"address_norm": "string"}
):
    for address in chunk["address_norm"].dropna():

        tokens = set(str(address).split())

        for token in tokens:
            if token:
                address_token_counts[token] += 1
    del chunk
    gc.collect()

print("Unique address tokens:", len(address_token_counts))
print("\nMost common address tokens:")
for token, count in address_token_counts.most_common(40):
    print(f"{token:30} {count:,}")

Unique address tokens: 607900

Most common address tokens:
rd                             998,770
no                             916,457
st                             668,885
dr                             499,393
ave                            417,689
floor                          319,586
maharashtra                    319,026
city                           298,419
tx                             292,289
nagar                          286,965
1                              281,622
a                              265,469
delhi                          262,478
c                              254,392
ln                             233,983
2                              228,483
ny                             225,422
pradesh                        214,708
new                            210,472
nc                             207,687
west                           200,360
oh                             187,361
h                              186,400
b                              183,628
plot 

In [62]:
frequency_buckets_address = {
    "1": 0,
    "2-5": 0,
    "6-10": 0,
    "11-50": 0,
    "51-100": 0,
    "101-500": 0,
    "501-1000": 0,
    "1001-5000": 0,
    "5001-10000": 0,
    "10001-50000": 0,
    "50001+": 0
}

for token, count in address_token_counts.items():
    if count == 1:
        frequency_buckets_address["1"] += 1
    elif count <= 5:
        frequency_buckets_address["2-5"] += 1
    elif count <= 10:
        frequency_buckets_address["6-10"] += 1
    elif count <= 50:
        frequency_buckets_address["11-50"] += 1
    elif count <= 100:
        frequency_buckets_address["51-100"] += 1
    elif count <= 500:
        frequency_buckets_address["101-500"] += 1
    elif count <= 1000:
        frequency_buckets_address["501-1000"] += 1
    elif count <= 5000:
        frequency_buckets_address["1001-5000"] += 1
    elif count <= 10000:
        frequency_buckets_address["5001-10000"] += 1
    elif count <= 50000:
        frequency_buckets_address["10001-50000"] += 1
    else:
        frequency_buckets_address["50001+"] += 1

for bucket, count in frequency_buckets_address.items():
    print(f"{bucket:>12}: {count:,}")

           1: 197,573
         2-5: 253,143
        6-10: 57,323
       11-50: 65,176
      51-100: 13,387
     101-500: 15,221
    501-1000: 2,798
   1001-5000: 2,500
  5001-10000: 348
 10001-50000: 317
      50001+: 114


In [63]:
MIN_ADDRESS_TOKEN_FREQ = 2
MAX_ADDRESS_TOKEN_FREQ = 500

allowed_address_tokens = {
    token
    for token, count in address_token_counts.items()
    if MIN_ADDRESS_TOKEN_FREQ <= count <= MAX_ADDRESS_TOKEN_FREQ
}

print("Allowed address tokens:", len(allowed_address_tokens))

Allowed address tokens: 404250


In [64]:
from collections import defaultdict
import gc
import pandas as pd

def build_address_token_index(path, allowed_tokens):
    index = defaultdict(list)
    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "address_norm"],
        chunksize=200_000,
        dtype={
            "entity_id": "string",
            "address_norm": "string"
        }
    ):
        for entity_id, address in zip(
            chunk["entity_id"],
            chunk["address_norm"]
        ):
            if pd.isna(address):
                continue

            tokens = set(str(address).split())

            for token in tokens:
                if token in allowed_tokens:
                    index[token].append(entity_id)

        del chunk
        gc.collect()

    return index

In [65]:
print("Building S2 address-token index...")
s2_address_token_index = build_address_token_index(S2_PATH, allowed_address_tokens)

print("S2 address-token blocks:", len(s2_address_token_index))

Building S2 address-token index...
S2 address-token blocks: 404250


In [66]:
def address_token_candidate_stats(
    s1_df,
    address_token_index,
    allowed_tokens,
    sample_size=10000
):
    candidate_counts = []

    for i, row in enumerate(s1_df.itertuples(index=False)):

        if i >= sample_size:
            break

        address = row.address_norm

        if pd.isna(address):
            candidate_counts.append(0)
            continue

        tokens = set(str(address).split())

        candidate_set = set()

        for token in tokens:
            if token in allowed_tokens:
                candidate_set.update(
                    address_token_index.get(token, [])
                )

        candidate_counts.append(len(candidate_set))

    s = pd.Series(candidate_counts)

    return {
        "sample_size": len(s),
        "mean": s.mean(),
        "median": s.median(),
        "p90": s.quantile(0.90),
        "p95": s.quantile(0.95),
        "p99": s.quantile(0.99),
        "max": s.max(),
        "zero_candidates": (s == 0).sum()
    }

In [67]:
S1_ADDRESS_PATH = S1_PATH

s1_address = pd.read_csv(
    S1_ADDRESS_PATH,
    sep="\t",
    usecols=[
        "entity_id",
        "country_norm",
        "address_norm"
    ],
    dtype={
        "entity_id": "string",
        "country_norm": "string",
        "address_norm": "string"
    }
)

print(s1_address.shape)
print(s1_address.columns.tolist())

(2206821, 3)
['entity_id', 'address_norm', 'country_norm']


In [68]:
s2_address_token_stats = address_token_candidate_stats(
    s1_address,
    s2_address_token_index,
    allowed_address_tokens,
    sample_size=10000
)

s2_address_token_stats

{'sample_size': 10000,
 'mean': np.float64(203.0828),
 'median': 131.5,
 'p90': np.float64(498.10000000000036),
 'p95': np.float64(641.0),
 'p99': np.float64(887.0200000000004),
 'max': 1794,
 'zero_candidates': np.int64(2228)}

In [69]:
def calculate_address_token_recall(
    s1_df,
    gt_lookup,
    address_token_index,
    allowed_tokens,
    prefix
):
    total_matches = 0
    captured_matches = 0

    for row in s1_df.itertuples(index=False):

        s1_id = row.entity_id
        address = row.address_norm

        true_ids = {
            x for x in gt_lookup.get(s1_id, set())
            if x.startswith(prefix)
        }

        if not true_ids:
            continue

        total_matches += len(true_ids)

        if pd.isna(address):
            continue

        tokens = set(str(address).split())

        candidate_set = set()

        for token in tokens:
            if token in allowed_tokens:
                candidate_set.update(
                    address_token_index.get(token, [])
                )

        captured_matches += len(
            true_ids.intersection(candidate_set)
        )

    recall = (
        captured_matches / total_matches
        if total_matches else 0
    )

    return total_matches, captured_matches, recall

In [70]:
s2_addr_token_total, s2_addr_token_captured, s2_addr_token_recall = \
    calculate_address_token_recall(
        s1_address,
        gt_lookup,
        s2_address_token_index,
        allowed_address_tokens,
        "S2-"
    )

print("S2 total true matches:", s2_addr_token_total)
print("S2 captured matches:", s2_addr_token_captured)
print(f"S2 address-token blocking recall: {s2_addr_token_recall:.4%}")

S2 total true matches: 3693619
S2 captured matches: 2523099
S2 address-token blocking recall: 68.3097%


In [33]:
s1_blocking = pd.read_csv(
    S1_PATH,
    sep="\t",
    usecols=[
        "entity_id",
        "country_norm",
        "name_core",
        "address_norm"
    ],
    dtype={
        "entity_id": "string",
        "country_norm": "string",
        "name_core": "string",
        "address_norm": "string"
    }
)

print(s1_blocking.shape)
print(s1_blocking.columns.tolist())

(2206821, 4)
['entity_id', 'address_norm', 'country_norm', 'name_core']


In [71]:
def calculate_three_way_union_recall(
    s1_df,
    gt_lookup,
    name_index,
    token_index,
    address_token_index,
    allowed_name_tokens,
    allowed_address_tokens,
    prefix
):
    total_matches = 0
    captured_matches = 0

    for row in s1_df.itertuples(index=False):

        s1_id = row.entity_id
        country = row.country_norm
        name = row.name_core
        address = row.address_norm

        true_ids = {
            x for x in gt_lookup.get(s1_id, set())
            if x.startswith(prefix)
        }

        if not true_ids:
            continue

        total_matches += len(true_ids)

        candidate_set = set()

        # BLOCKER 1: Exact country + name
        if pd.notna(name):
            candidate_set.update(
                name_index.get((country, name), [])
            )

        # BLOCKER 2: Name tokens
        if pd.notna(name):

            name_tokens = set(
                meaningful_tokens(name)
            )

            for token in name_tokens:

                if token in allowed_name_tokens:
                    candidate_set.update(
                        token_index.get(token, [])
                    )

        # BLOCKER 3: Address tokens
        if pd.notna(address):

            address_tokens = set(
                str(address).split()
            )

            for token in address_tokens:

                if token in allowed_address_tokens:
                    candidate_set.update(
                        address_token_index.get(token, [])
                    )

        captured_matches += len(
            true_ids.intersection(candidate_set)
        )

    recall = (
        captured_matches / total_matches
        if total_matches else 0
    )

    return total_matches, captured_matches, recall

In [72]:
s2_all_total, s2_all_captured, s2_all_recall = \
    calculate_three_way_union_recall(
        s1_blocking,
        gt_lookup,
        s2_name_index,
        s2_token_index,
        s2_address_token_index,
        allowed_tokens,
        allowed_address_tokens,
        "S2-"
    )

print("S2 total true matches:", s2_all_total)
print("S2 captured matches:", s2_all_captured)
print(f"S2 three-way blocking recall: {s2_all_recall:.4%}")

S2 total true matches: 3693619
S2 captured matches: 3222405
S2 three-way blocking recall: 87.2425%


In [73]:
def find_missed_s2_matches(
    s1_df,
    gt_lookup,
    name_index,
    token_index,
    address_token_index,
    allowed_name_tokens,
    allowed_address_tokens,
    sample_limit=100
):
    missed = []

    for row in s1_df.itertuples(index=False):

        s1_id = row.entity_id
        country = row.country_norm
        name = row.name_core
        address = row.address_norm

        true_ids = {
            x for x in gt_lookup.get(s1_id, set())
            if x.startswith("S2-")
        }

        if not true_ids:
            continue

        candidate_set = set()
        if pd.notna(name):
            candidate_set.update(
                name_index.get((country, name), [])
            )
        if pd.notna(name):
            for token in set(meaningful_tokens(name)):
                if token in allowed_name_tokens:
                    candidate_set.update(
                        token_index.get(token, [])
                    )
        if pd.notna(address):
            for token in set(str(address).split()):
                if token in allowed_address_tokens:
                    candidate_set.update(
                        address_token_index.get(token, [])
                    )

        missed_ids = true_ids - candidate_set

        for s2_id in missed_ids:

            missed.append({
                "s1_id": s1_id,
                "s2_id": s2_id
            })

            if len(missed) >= sample_limit:
                return missed

    return missed

In [74]:
# Taking 100 missing S2 examples:
missed_s2 = find_missed_s2_matches(
    s1_blocking,
    gt_lookup,
    s2_name_index,
    s2_token_index,
    s2_address_token_index,
    allowed_tokens,
    allowed_address_tokens,
    sample_limit=100
)

print("Missed examples collected:", len(missed_s2))
missed_s2[:10]

Missed examples collected: 100


[{'s1_id': 'S1-133037285', 's2_id': 'S2-407207105'},
 {'s1_id': 'S1-27541239', 's2_id': 'S2-199341916'},
 {'s1_id': 'S1-629417405', 's2_id': 'S2-928426462'},
 {'s1_id': 'S1-504790211', 's2_id': 'S2-340882825'},
 {'s1_id': 'S1-865131206', 's2_id': 'S2-657355663'},
 {'s1_id': 'S1-626914593', 's2_id': 'S2-563003232'},
 {'s1_id': 'S1-597762257', 's2_id': 'S2-152865025'},
 {'s1_id': 'S1-597762257', 's2_id': 'S2-750318376'},
 {'s1_id': 'S1-597762257', 's2_id': 'S2-535895394'},
 {'s1_id': 'S1-252242682', 's2_id': 'S2-84842746'}]

In [75]:
missed_s1_ids = {
    x["s1_id"]
    for x in missed_s2}
missed_s2_ids = {
    x["s2_id"]
    for x in missed_s2
}

In [76]:
s1_missed_sample = s1_blocking[
    s1_blocking["entity_id"].isin(missed_s1_ids)
].copy()

In [77]:
s2_missed_sample = []

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country_norm",
        "name_core",
        "address_norm"
    ],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "business_name": "string",
        "business_address": "string",
        "country_norm": "string",
        "name_core": "string",
        "address_norm": "string"
    }
):
    matched = chunk[
        chunk["entity_id"].isin(missed_s2_ids)
    ]

    if len(matched):
        s2_missed_sample.append(matched)

    del chunk
    gc.collect()

s2_missed_sample = pd.concat(
    s2_missed_sample,
    ignore_index=True
)

print("S2 missed records retrieved:", len(s2_missed_sample))

S2 missed records retrieved: 100


In [78]:
missed_pairs = pd.DataFrame(missed_s2)
missed_pairs = (
    missed_pairs
    .merge(
        s1_missed_sample,
        left_on="s1_id",
        right_on="entity_id",
        how="left",
        suffixes=("_s1", "")
    )
    .drop(columns=["entity_id"])
)

missed_pairs = (
    missed_pairs
    .merge(
        s2_missed_sample,
        left_on="s2_id",
        right_on="entity_id",
        how="left",
        suffixes=("_s1", "_s2")
    )
    .drop(columns=["entity_id"])
)

In [79]:
print(missed_pairs.columns.tolist())

['s1_id', 's2_id', 'address_norm_s1', 'country_norm_s1', 'name_core_s1', 'business_name', 'business_address', 'address_norm_s2', 'country_norm_s2', 'name_core_s2']


In [80]:
display(
    missed_pairs[
        [
            "s1_id",
            "s2_id",
            "business_name",
            "business_address",
            "name_core_s1",
            "name_core_s2",
            "address_norm_s1",
            "address_norm_s2"
        ]
    ].head(50)
)

,s1_id,s2_id,business_name,business_address,name_core_s1,name_core_s2,address_norm_s1,address_norm_s2
0,S1-133037285,S2-407207105,Christ [Chape1],"2100 CAMERON DR, DUNDALK, MD",christ chapel,christ chape1,2100 cameron dr unit apt g dundalk md,2100 cameron dr dundalk md
1,S1-27541239,S2-199341916,NEXUS ACORMHR RAIN,"CHURCH STREET, NASHVILLE, TN",nexus anchor rain,nexus acormhr rain,1111 church st unit 2007 nashville tn,church st nashville tn
2,S1-629417405,S2-928426462,Moore Inc Center,"337 OAKLAND AVE, MICHHIGAN CITY CITY, IN",moore bitwise,moore inc center,337 oakland ave michigan city in,337 oakland ave michhigan city city in
3,S1-504790211,S2-340882825,Crystal PC Lending,<NA>,crystal lending pc,crystal pc lending,11643 prosperity rd south jordan ut,<NA>
4,S1-865131206,S2-657355663,HELI0S LP,"66 EDGEWOOD ST, BRIDGEPORT, CT",helios,heli0s lp,66 edgewood st bridgeport ct,66 edgewood st bridgeport ct
5,S1-626914593,S2-563003232,Unified Choice Dynamx,"MIDDLE RIVER, 2701 Eastern Blvd, MD",unified choice dynamix,unified choice dynamx,unit bldg 3030 md 2701 eastern blvd middle river,middle river 2701 eastern blvd md
6,S1-597762257,S2-152865025,ग्रीन लॉजिस्टिक्स प्राइवेट लिमिटेड,"E-7, NEW DELHI, SECOND FLOOR, SOUTH DELHI, Delhi",green logistics,ग र न ल ज स ट क स प र इव ट ल म ट ड,e 7 second floor new delhi south delhi delhi,e 7 new delhi second floor south delhi delhi
7,S1-597762257,S2-750318376,ग्रीन लॉजिस्टिक्स प्राइवेट लिमिटेड,"E-7, SECOND FLOOR, NEW DELHI, SOUTH DELHI, Delhi",green logistics,ग र न ल ज स ट क स प र इव ट ल म ट ड,e 7 second floor new delhi south delhi delhi,e 7 second floor new delhi south delhi delhi
8,S1-597762257,S2-535895394,ग्रीन लॉजिस्टिक्स प्राइवेट लिमिटेड,"E-7, SECOND FLOOR, NEW DELHI, SOUTH DELHI, दिल्ली",green logistics,ग र न ल ज स ट क स प र इव ट ल म ट ड,e 7 second floor new delhi south delhi delhi,e 7 second floor new delhi south delhi द ल ल
9,S1-252242682,S2-84842746,*** aggieenagle.com,"54ND AVENUE, MINNEAPLIS, MN",aggie e nagle l c s w,aggieenagle com,11900 54th ave plymouth mn,54nd ave minneaplis mn


In [81]:
from difflib import SequenceMatcher
import pandas as pd

def char_similarity(a, b):
    if pd.isna(a) or pd.isna(b):
        return None

    a = str(a)
    b = str(b)

    if not a or not b:
        return None
    return SequenceMatcher(None, a, b).ratio()

missed_pairs["name_char_similarity"] = missed_pairs.apply(
    lambda row: char_similarity(
        row["name_core_s1"],
        row["name_core_s2"]
    ),
    axis=1
)

missed_pairs["address_char_similarity"] = missed_pairs.apply(
    lambda row: char_similarity(
        row["address_norm_s1"],
        row["address_norm_s2"]
    ),
    axis=1
)

print("Done.")

Done.


In [82]:
print("Name Character Similarity")
print(
    missed_pairs["name_char_similarity"]
    .describe()
)

print("\nAddress Character Similarity")
print(
    missed_pairs["address_char_similarity"]
    .describe()
)

Name Character Similarity
count    100.000000
mean       0.586463
std        0.347140
min        0.039216
25%        0.153846
50%        0.750000
75%        0.868750
max        0.979592
Name: name_char_similarity, dtype: float64

Address Character Similarity
count    89.000000
mean      0.778370
std       0.186548
min       0.305085
25%       0.661538
50%       0.825397
75%       0.927152
max       1.000000
Name: address_char_similarity, dtype: float64


In [83]:
bins = [0, 0.5, 0.6, 0.7, 0.8, 0.9, 1.01]
labels = [
    "<0.5",
    "0.5-0.6",
    "0.6-0.7",
    "0.7-0.8",
    "0.8-0.9",
    "0.9-1.0"
]

name_buckets = pd.cut(
    missed_pairs["name_char_similarity"],
    bins=bins,
    labels=labels,
    right=False
)

print(name_buckets.value_counts().sort_index())

name_char_similarity
<0.5       32
0.5-0.6     6
0.6-0.7     7
0.7-0.8    14
0.8-0.9    19
0.9-1.0    22
Name: count, dtype: int64


In [84]:
display(
    missed_pairs[
        [
            "name_core_s1",
            "name_core_s2",
            "name_char_similarity",
            "address_norm_s1",
            "address_norm_s2",
            "address_char_similarity"
        ]
    ]
    .sort_values("name_char_similarity")
    .reset_index(drop=True)
)

,name_core_s1,name_core_s2,name_char_similarity,address_norm_s1,address_norm_s2,address_char_similarity
0,creative services,క ర య ట వ సర వ స స ప ర వ ట ల మ ట డ,0.039216,h no 8 3 214 1 a 1 plot no b2madhura nagar hyd...,hyderabad andhra pradesh telangana 8 3 214 1 a 1,0.544000
1,creative services,క ర య ట వ సర వ స స ప ర వ ట ల మ ట డ,0.039216,h no 8 3 214 1 a 1 plot no b2madhura nagar hyd...,h no 8 3 214 1 a 1 andhra pradesh hyderabad te...,0.661538
2,green logistics,ग र न ल ज स ट क स प र इव ट ल म ट ड,0.040816,e 7 second floor new delhi south delhi delhi,e 7 new delhi second floor south delhi delhi,0.772727
3,green logistics,ग र न ल ज स ट क स प र इव ट ल म ट ड,0.040816,e 7 second floor new delhi south delhi delhi,e 7 second floor new delhi south delhi delhi,1.000000
4,green logistics,ग र न ल ज स ट क स प र इव ट ल म ट ड,0.040816,e 7 second floor new delhi south delhi delhi,e 7 second floor new delhi south delhi द ल ल,0.886364
...,...,...,...,...,...,...
95,gulf apex,gulf alpex,0.947368,1025 holland sylvania rd unit 5 toledo oh,oh 1025 holland sylvania rd toledo,0.826667
96,highland learning center,highland learning centre,0.958333,1240 indiana ave salt lake city ut,1240 indiana ave ut salt lake city,0.911765
97,unified green target,unified green tbarget,0.975610,1037 cedar grove rd halifax county va,halifax county 1037 cedar grove rd va,0.594595
98,unified choice dynamix,unified choice dynamx,0.976744,unit bldg 3030 md 2701 eastern blvd middle river,middle river 2701 eastern blvd md,0.567901


In [85]:
thresholds = [0.60, 0.70, 0.80, 0.85, 0.90, 0.95]
for threshold in thresholds:
    count = (missed_pairs["name_char_similarity"] >= threshold).sum()

    print(
        f"Similarity >= {threshold:.2f}: "
        f"{count}/100 ({count:.1f}%)"
    )

Similarity >= 0.60: 62/100 (62.0%)
Similarity >= 0.70: 55/100 (55.0%)
Similarity >= 0.80: 41/100 (41.0%)
Similarity >= 0.85: 30/100 (30.0%)
Similarity >= 0.90: 22/100 (22.0%)
Similarity >= 0.95: 4/100 (4.0%)


Name vs Address Similarity

In [86]:
def token_jaccard(a, b):
    if pd.isna(a) or pd.isna(b):
        return None

    a_tokens = set(str(a).split())
    b_tokens = set(str(b).split())

    if not a_tokens or not b_tokens:
        return None

    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)
missed_pairs["name_token_jaccard"] = missed_pairs.apply(
    lambda row: token_jaccard(
        row["name_core_s1"],
        row["name_core_s2"]
    ),
    axis=1
)

missed_pairs["address_token_jaccard"] = missed_pairs.apply(
    lambda row: token_jaccard(
        row["address_norm_s1"],
        row["address_norm_s2"]
    ),
    axis=1
)

In [87]:
print("NAME TOKEN JACCARD")
print(missed_pairs["name_token_jaccard"].describe())

print("\nADDRESS TOKEN JACCARD")
print(missed_pairs["address_token_jaccard"].describe())

NAME TOKEN JACCARD
count    100.000000
mean       0.320810
std        0.293351
min        0.000000
25%        0.000000
50%        0.333333
75%        0.500000
max        1.000000
Name: name_token_jaccard, dtype: float64

ADDRESS TOKEN JACCARD
count    89.000000
mean      0.695194
std       0.225127
min       0.210526
25%       0.538462
50%       0.666667
75%       0.888889
max       1.000000
Name: address_token_jaccard, dtype: float64


In [88]:
print("NAME TOKEN JACCARD")

print(
    pd.cut(
        missed_pairs["name_token_jaccard"],
        bins=[0, 0.2, 0.4, 0.6, 0.8, 1.01],
        labels=[
            "<0.2",
            "0.2-0.4",
            "0.4-0.6",
            "0.6-0.8",
            "0.8-1.0"
        ],
        right=False
    ).value_counts().sort_index()
)

print("\nADDRESS TOKEN JACCARD")

print(
    pd.cut(
        missed_pairs["address_token_jaccard"],
        bins=[0, 0.2, 0.4, 0.6, 0.8, 1.01],
        labels=[
            "<0.2",
            "0.2-0.4",
            "0.4-0.6",
            "0.6-0.8",
            "0.8-1.0"
        ],
        right=False
    ).value_counts().sort_index()
)

NAME TOKEN JACCARD
name_token_jaccard
<0.2       37
0.2-0.4    21
0.4-0.6    22
0.6-0.8    14
0.8-1.0     6
Name: count, dtype: int64

ADDRESS TOKEN JACCARD
address_token_jaccard
<0.2        0
0.2-0.4     7
0.4-0.6    24
0.6-0.8    27
0.8-1.0    31
Name: count, dtype: int64


In [89]:
display(
    missed_pairs[
        [
            "name_core_s1",
            "name_core_s2",
            "name_char_similarity",
            "name_token_jaccard",
            "address_char_similarity",
            "address_token_jaccard"
        ]
    ]
    .sort_values(
        ["address_token_jaccard", "name_char_similarity"],
        ascending=False
    )
    .reset_index(drop=True)
)

,name_core_s1,name_core_s2,name_char_similarity,name_token_jaccard,address_char_similarity,address_token_jaccard
0,unified green target,unified green tbarget,0.975610,0.500000,0.594595,1.0
1,highland learning center,highland learning centre,0.958333,0.500000,0.911765,1.0
2,anand brothers,anand 8rothers,0.928571,0.333333,0.927152,1.0
3,cannon precious,cann0n précious,0.866667,0.000000,0.645161,1.0
4,anand brothers,anandbrothers com,0.838710,0.000000,0.768212,1.0
...,...,...,...,...,...,...
95,clean investment services,clean investment center,0.791667,0.500000,NaN,NaN
96,jobst royal metal works,jobst royal llc works metal,0.760000,0.800000,NaN,NaN
97,classic printing enterprises,classic enterprises corp services,0.622951,0.400000,NaN,NaN
98,paul holdings,paul,0.470588,0.500000,NaN,NaN


In [90]:
# Filling the missing similarities with 0 (experiment)
experiment_df = missed_pairs.copy()

for col in [
    "name_char_similarity",
    "name_token_jaccard",
    "address_char_similarity",
    "address_token_jaccard"
]:
    experiment_df[col] = experiment_df[col].fillna(0)

# Address is currently the strongest signal
experiment_df["combined_similarity"] = (
    0.25 * experiment_df["name_char_similarity"] +
    0.15 * experiment_df["name_token_jaccard"] +
    0.25 * experiment_df["address_char_similarity"] +
    0.35 * experiment_df["address_token_jaccard"]
)

display(
    experiment_df[
        [
            "name_core_s1",
            "name_core_s2",
            "name_char_similarity",
            "name_token_jaccard",
            "address_char_similarity",
            "address_token_jaccard",
            "combined_similarity"
        ]
    ]
    .sort_values("combined_similarity", ascending=False)
    .reset_index(drop=True)
)

,name_core_s1,name_core_s2,name_char_similarity,name_token_jaccard,address_char_similarity,address_token_jaccard,combined_similarity
0,highland learning center,highland learning,0.829268,0.666667,1.000000,1.000000,0.907317
1,highland learning center,highland learning centre,0.958333,0.500000,0.911765,1.000000,0.892525
2,shiv technologies,dr shiv technologies,0.918919,0.666667,0.919355,0.900000,0.874568
3,anand brothers,anand 8rothers,0.928571,0.333333,0.927152,1.000000,0.863931
4,apex,inc apex,0.666667,0.500000,1.000000,1.000000,0.841667
...,...,...,...,...,...,...,...
95,shakti consulting,शक त क सल ट ग ल म ट ड,0.052632,0.000000,0.463158,0.266667,0.222281
96,classic printing enterprises,classic enterprises corp services,0.622951,0.400000,0.000000,0.000000,0.215738
97,paul holdings,paul,0.470588,0.500000,0.000000,0.000000,0.192647
98,high estate,ह ई एस ट ट ल म ट ड,0.068966,0.000000,0.305085,0.210526,0.167197


In [91]:
print(experiment_df["combined_similarity"].describe())

count    100.000000
mean       0.584477
std        0.208372
min        0.157143
25%        0.389999
50%        0.636902
75%        0.751012
max        0.907317
Name: combined_similarity, dtype: float64


In [92]:
for threshold in [0.50, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85]:
    count = (experiment_df["combined_similarity"] >= threshold).sum()

    print(
        f"Combined similarity >= {threshold:.2f}: "
        f"{count}/100 ({count:.1f}%)"
    )

Combined similarity >= 0.50: 69/100 (69.0%)
Combined similarity >= 0.60: 57/100 (57.0%)
Combined similarity >= 0.65: 48/100 (48.0%)
Combined similarity >= 0.70: 38/100 (38.0%)
Combined similarity >= 0.75: 27/100 (27.0%)
Combined similarity >= 0.80: 15/100 (15.0%)
Combined similarity >= 0.85: 4/100 (4.0%)


Character N-Grams

In [93]:
def char_ngrams(text, n=3):
    if pd.isna(text):
        return set()

    text = str(text).replace(" ", "")

    if len(text) < n:
        return {text}

    return {
        text[i:i+n]
        for i in range(len(text) - n + 1)
    }


def ngram_jaccard(a, b, n=3):
    a_grams = char_ngrams(a, n)
    b_grams = char_ngrams(b, n)

    if not a_grams or not b_grams:
        return 0.0

    return len(a_grams & b_grams) / len(a_grams | b_grams)

In [94]:
missed_pairs["name_char3_jaccard"] = missed_pairs.apply(
    lambda row: ngram_jaccard(
        row["name_core_s1"],
        row["name_core_s2"],
        n=3
    ),
    axis=1)

missed_pairs["name_char4_jaccard"] = missed_pairs.apply(
    lambda row: ngram_jaccard(
        row["name_core_s1"],
        row["name_core_s2"],
        n=4
    ),
    axis=1)

In [95]:
print("CHAR 3-GRAM JACCARD")
print(missed_pairs["name_char3_jaccard"].describe())
print("\nCHAR 4-GRAM JACCARD")
print(missed_pairs["name_char4_jaccard"].describe())

CHAR 3-GRAM JACCARD
count    100.000000
mean       0.385566
std        0.308936
min        0.000000
25%        0.000000
50%        0.433036
75%        0.630515
max        0.950000
Name: name_char3_jaccard, dtype: float64

CHAR 4-GRAM JACCARD
count    100.000000
mean       0.338205
std        0.299855
min        0.000000
25%        0.000000
50%        0.311422
75%        0.555556
max        0.947368
Name: name_char4_jaccard, dtype: float64


Threshold Experiment

In [96]:
for column in [
    "name_char3_jaccard",
    "name_char4_jaccard"]:
    print(f"\n{column}")

    for threshold in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
        count = (
            missed_pairs[column] >= threshold
        ).sum()

        print(
            f">= {threshold:.2f}: "
            f"{count}/100 ({count}%)"
        )


name_char3_jaccard
>= 0.20: 67/100 (67%)
>= 0.30: 57/100 (57%)
>= 0.40: 52/100 (52%)
>= 0.50: 45/100 (45%)
>= 0.60: 31/100 (31%)
>= 0.70: 19/100 (19%)
>= 0.80: 12/100 (12%)

name_char4_jaccard
>= 0.20: 58/100 (58%)
>= 0.30: 52/100 (52%)
>= 0.40: 46/100 (46%)
>= 0.50: 34/100 (34%)
>= 0.60: 22/100 (22%)
>= 0.70: 15/100 (15%)
>= 0.80: 11/100 (11%)


In [97]:
display(
    missed_pairs[
        [
            "name_core_s1",
            "name_core_s2",
            "name_char_similarity",
            "name_token_jaccard",
            "name_char3_jaccard",
            "name_char4_jaccard"
        ]
    ]
    .sort_values("name_char3_jaccard")
    .reset_index(drop=True)
)

,name_core_s1,name_core_s2,name_char_similarity,name_token_jaccard,name_char3_jaccard,name_char4_jaccard
0,green logistics,ग र न ल ज स ट क स प र इव ट ल म ट ड,0.040816,0.000000,0.000000,0.000000
1,green logistics,ग र न ल ज स ट क स प र इव ट ल म ट ड,0.040816,0.000000,0.000000,0.000000
2,corner hypnosis,brixlyra,0.173913,0.000000,0.000000,0.000000
3,green logistics,ग र न ल ज स ट क स प र इव ट ल म ट ड,0.040816,0.000000,0.000000,0.000000
4,shiv technologies,ಶ ವ ಟ ಕ ನ ಲಜ ಸ ಎಲ ಎಲ ಪ,0.051282,0.000000,0.000000,0.000000
...,...,...,...,...,...,...
95,shiv technologies,dr shiv technologies,0.918919,0.666667,0.875000,0.866667
96,pacific learning laboratories,pacific learning laboratories l l c,0.906250,0.600000,0.892857,0.888889
97,operating engineers union local no 269,operating engineers union local no,0.944444,0.833333,0.903226,0.900000
98,adams williams and alexander carolina,adams williams and alexander carolina l l c,0.925000,0.714286,0.906250,0.909091


3-GRAM is better than 4-GRAM

In [98]:
char3_counter = Counter()

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):

    for name in chunk["name_core"].dropna():
        grams = char_ngrams(name, n=3)

        char3_counter.update(grams)

    del chunk
    gc.collect()

print("Unique 3-grams:", len(char3_counter))

Unique 3-grams: 90023


In [99]:
char3_freq = pd.Series(char3_counter)
print(char3_freq.describe())

count     90023.000000
mean        901.906091
std        7896.146677
min           1.000000
25%           2.000000
50%          10.000000
75%          69.000000
max      509272.000000
dtype: float64


In [100]:
bins = [
    1,
    2,
    5,
    10,
    20,
    50,
    100,
    500,
    1000,
    5000,
    10000,
    float("inf")
]

labels = [
    "1",
    "2-4",
    "5-9",
    "10-19",
    "20-49",
    "50-99",
    "100-499",
    "500-999",
    "1000-4999",
    "5000-9999",
    "10000+"
]

print(
    pd.cut(
        char3_freq,
        bins=bins,
        labels=labels,
        right=False,
        include_lowest=True
    ).value_counts().sort_index()
)

1            15716
2-4          17405
5-9          11145
10-19         9223
20-49        10715
50-99         6517
100-499      10463
500-999       2692
1000-4999     3598
5000-9999      995
10000+        1554
Name: count, dtype: int64


In [101]:
print(char3_freq.sort_values(ascending=False).head(30))

ent    509272
ter    458539
ers    436935
nte    382223
ion    380764
ing    367878
ate    292581
com    285833
tio    266353
ice    260997
art    258857
and    255983
ati    242251
ner    236182
ser    230001
cen    228713
llc    223707
rvi    220140
dia    218173
erv    217787
vic    217091
ons    209638
ind    208741
par    200443
tal    199241
लमट    190106
मटड    190091
din    188122
str    185234
ces    183961
dtype: int64


In [102]:
allowed_char3 = {
    gram
    for gram, freq in char3_counter.items()
    if 2 <= freq <= 500}
print("Allowed 3-grams:", len(allowed_char3))

Allowed 3-grams: 65473


In [103]:
from collections import defaultdict
import gc
import pandas as pd

char3_index = defaultdict(list)

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):

    for entity_id, name in zip(
        chunk["entity_id"],
        chunk["name_core"]
    ):

        if pd.isna(name):
            continue

        for gram in char_ngrams(name, n=3):
            if gram in allowed_char3:
                char3_index[gram].append(entity_id)

    del chunk
    gc.collect()
print("Index blocks:", len(char3_index))

Index blocks: 65473


In [104]:
candidate_counts = []

for row in s1_blocking.itertuples(index=False):
    name = row.name_core
    if pd.isna(name):
        candidate_counts.append(0)
        continue
    candidate_set = set()
    for gram in char_ngrams(name, n=3):
        if gram in char3_index:
            candidate_set.update(char3_index[gram])
    candidate_counts.append(len(candidate_set))
    if len(candidate_counts) >= 100_000:
        break
candidate_counts = pd.Series(candidate_counts)

print(candidate_counts.describe())
print("Zero candidates:", (candidate_counts == 0).sum())
print("Candidates > 1000:", (candidate_counts > 1000).sum())
print("Candidates > 5000:", (candidate_counts > 5000).sum())

count    100000.000000
mean         46.891170
std         140.894528
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        2549.000000
dtype: float64
Zero candidates: 85970
Candidates > 1000: 131
Candidates > 5000: 0


In [105]:
def calculate_char3_recall(
    s1_df,
    gt_lookup,
    char3_index,
    allowed_char3,
    prefix
):
    total_matches = 0
    captured_matches = 0

    for row in s1_df.itertuples(index=False):
        s1_id = row.entity_id
        name = row.name_core
        true_ids = {
            x for x in gt_lookup.get(s1_id, set())
            if x.startswith(prefix)
        }

        if not true_ids:
            continue

        total_matches += len(true_ids)
        candidate_set = set()
        if pd.notna(name) and name != "":
            for gram in char_ngrams(name, n=3):
                if gram in allowed_char3:
                    candidate_set.update(
                        char3_index.get(gram, [])
                    )

        captured_matches += len(
            true_ids.intersection(candidate_set)
        )
    recall = (
        captured_matches / total_matches
        if total_matches
        else 0)

    return total_matches, captured_matches, recall

In [106]:
s2_c3_total, s2_c3_captured, s2_c3_recall = calculate_char3_recall(
    s1_blocking,
    gt_lookup,
    char3_index,
    allowed_char3,
    "S2-")

print("S2")
print("Total true matches :", s2_c3_total)
print("Captured matches   :", s2_c3_captured)
print(f"3-gram recall      : {s2_c3_recall:.4%}")

S2
Total true matches : 3693619
Captured matches   : 439775
3-gram recall      : 11.9063%


In [107]:
print([x for x in [
        "s2_name_index",
        "token_index",
        "address_token_index",
        "allowed_name_tokens",
        "allowed_address_tokens"
    ] if x in globals()]
)

['s2_name_index', 'allowed_address_tokens']


In [108]:
from collections import Counter, defaultdict
import gc
import pandas as pd

name_token_counter = Counter()
for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):

    for name in chunk["name_core"].dropna():
        name_token_counter.update(
            set(meaningful_tokens(name)))

    del chunk
    gc.collect()
print("Unique name tokens:", len(name_token_counter))

Unique name tokens: 810209


In [109]:
allowed_name_tokens = {token
    for token, freq in name_token_counter.items()
    if 2 <= freq <= 500}

print("Allowed name tokens: ", len(allowed_name_tokens))

Allowed name tokens:  167295


In [110]:
name_token_index = defaultdict(list)
for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):

    for entity_id, name in zip(
        chunk["entity_id"],
        chunk["name_core"]):

        if pd.isna(name):
            continue

        for token in set(meaningful_tokens(name)):
            if token in allowed_name_tokens:
                name_token_index[token].append(entity_id)

    del chunk
    gc.collect()

print("Name-token blocks:", len(name_token_index))

Name-token blocks: 167295


In [111]:
address_token_index = defaultdict(list)
for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["entity_id", "address_norm"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "address_norm": "string"
    }
):

    for entity_id, address in zip(
        chunk["entity_id"],
        chunk["address_norm"]
    ):

        if pd.isna(address):
            continue

        for token in set(str(address).split()):
            if token in allowed_address_tokens:
                address_token_index[token].append(entity_id)
    del chunk
    gc.collect()

print("Address-token blocks:", len(address_token_index))

Address-token blocks: 404250


In [112]:
missed_s2_ids = set()
total_true = 0
captured_existing = 0

for row in s1_blocking.itertuples(index=False):
    s1_id = row.entity_id
    country = row.country_norm
    name = row.name_core
    address = row.address_norm

    true_ids = {
        x for x in gt_lookup.get(s1_id, set())
        if x.startswith("S2-")
    }

    if not true_ids:
        continue

    total_true += len(true_ids)
    candidate_set = set()

    # 1. Exact country + name
    if pd.notna(name) and name != "":
        candidate_set.update(
            s2_name_index.get((country, name), [])
        )

        # 2. Name token
        for token in set(meaningful_tokens(name)):
            if token in allowed_name_tokens:
                candidate_set.update(
                    name_token_index.get(token, [])
                )
    # 3. Address token
    if pd.notna(address) and address != "":
        for token in set(str(address).split()):
            if token in allowed_address_tokens:
                candidate_set.update(
                    address_token_index.get(token, [])
                )

    captured = true_ids.intersection(candidate_set)

    captured_existing += len(captured)

    missed_s2_ids.update(
        true_ids - captured
    )
print("Total true S2 matches:", total_true)
print("Captured by existing blocker:", captured_existing)
print("Missed by existing blocker:", len(missed_s2_ids))

Total true S2 matches: 3693619
Captured by existing blocker: 3222405
Missed by existing blocker: 471214


In [113]:
newly_recovered_by_char3 = 0

for row in s1_blocking.itertuples(index=False):

    s1_id = row.entity_id
    name = row.name_core

    true_ids = {
        x for x in gt_lookup.get(s1_id, set())
        if x.startswith("S2-")
    }

    if not true_ids or pd.isna(name) or name == "":
        continue
    missed_for_s1 = true_ids.intersection(missed_s2_ids)
    if not missed_for_s1:
        continue
    char3_candidates = set()
    for gram in char_ngrams(name, n=3):
        if gram in allowed_char3:
            char3_candidates.update(
                char3_index.get(gram, [])
            )

    newly_recovered_by_char3 += len(
        missed_for_s1.intersection(char3_candidates)
    )
print(
    "New true matches recovered by 3-gram:",
    newly_recovered_by_char3
)
print(
    "Percentage of existing misses recovered:",
    f"{newly_recovered_by_char3 / len(missed_s2_ids):.4%}"
)

New true matches recovered by 3-gram: 19126
Percentage of existing misses recovered: 4.0589%


Transliteration

In [114]:
import unicodedata
from collections import Counter
def get_script_name(text):
    scripts = set()
    if pd.isna(text):
        return scripts
    for ch in str(text):
        try:
            name = unicodedata.name(ch)
        except ValueError:
            continue

        for script in [
            "DEVANAGARI",
            "BENGALI",
            "GURMUKHI",
            "GUJARATI",
            "ORIYA",
            "TAMIL",
            "TELUGU",
            "KANNADA",
            "MALAYALAM",
            "SINHALA",
            "ARABIC",
            "BENGALI",
            "LATIN"
        ]:
            if script in name:
                scripts.add(script)

    return scripts

script_counts = Counter()

for chunk in pd.read_csv(
    S1_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):
    for name in chunk["name_core"].dropna():
        scripts = get_script_name(name)

        # Recording each non Latin script
        for script in scripts:
            if script != "LATIN":
                script_counts[script] += 1

    del chunk
    gc.collect()

print(script_counts)

Counter()


In [119]:
def detect_script(text):
    if pd.isna(text):
        return "NA"

    scripts = set()

    for ch in str(text):
        try:
            char_name = unicodedata.name(ch)
        except ValueError:
            continue

        if "LATIN" in char_name:
            scripts.add("LATIN")
        elif "DEVANAGARI" in char_name:
            scripts.add("DEVANAGARI")
        elif "GUJARATI" in char_name:
            scripts.add("GUJARATI")
        elif "KANNADA" in char_name:
            scripts.add("KANNADA")
        elif "TELUGU" in char_name:
            scripts.add("TELUGU")
        elif "TAMIL" in char_name:
            scripts.add("TAMIL")
        elif "MALAYALAM" in char_name:
            scripts.add("MALAYALAM")
        elif "BENGALI" in char_name:
            scripts.add("BENGALI")
        elif "GURMUKHI" in char_name:
            scripts.add("GURMUKHI")
        elif "ORIYA" in char_name:
            scripts.add("ORIYA")
        elif "ARABIC" in char_name:
            scripts.add("ARABIC")

    if not scripts:
        return "OTHER"

    return "+".join(sorted(scripts))


print("detect_script restored")

detect_script restored


In [120]:
script_counts_s1 = Counter()
examples_s1 = {}

for chunk in pd.read_csv(
    RAW_S1_PATH,
    sep="\t",
    usecols=["business_name"],
    chunksize=200_000,
    dtype={"business_name": "string"}
):
    for name in chunk["business_name"].dropna():
        script = detect_script(name)
        script_counts_s1[script] += 1

        if script != "LATIN" and script not in examples_s1:
            examples_s1[script] = name

    del chunk
    gc.collect()

print("S1 script counts:")
for script, count in script_counts_s1.most_common():
    print(f"{script:30s} {count:,}")

print("\nS1 non-Latin examples:")
for script, example in examples_s1.items():
    print(f"{script:30s} -> {example}")

S1 script counts:
LATIN                          2,206,821

S1 non-Latin examples:


In [121]:
script_counts_s2 = Counter()
examples_s2 = {}

for chunk in pd.read_csv(
    RAW_S2_PATH,
    sep="\t",
    usecols=["business_name"],
    chunksize=200_000,
    dtype={"business_name": "string"}
):
    for name in chunk["business_name"].dropna():
        script = detect_script(name)
        script_counts_s2[script] += 1

        if script != "LATIN" and script not in examples_s2:
            examples_s2[script] = name

    del chunk
    gc.collect()

print("S2 script counts:")
for script, count in script_counts_s2.most_common():
    print(f"{script:30s} {count:,}")

print("\nS2 non-Latin examples:")
for script, example in examples_s2.items():
    print(f"{script:30s} -> {example}")

S2 script counts:
LATIN                          4,560,075
DEVANAGARI                     258,863
TELUGU                         37,799
KANNADA                        35,787
TAMIL                          32,440
GUJARATI                       29,700
BENGALI                        29,497
MALAYALAM                      18,062
DEVANAGARI+LATIN               10,561
ORIYA                          7,194
GURMUKHI                       6,417
LATIN+TELUGU                   1,524
KANNADA+LATIN                  1,424
LATIN+TAMIL                    1,341
GUJARATI+LATIN                 1,229
BENGALI+LATIN                  1,226
LATIN+MALAYALAM                711
LATIN+ORIYA                    299
GURMUKHI+LATIN                 271
OTHER                          194

S2 non-Latin examples:
DEVANAGARI                     -> राम मार्केटिंग प्राइवेट लिमिटेड
TAMIL                          -> குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்
GUJARATI                       -> શક્તિ અર્બન પ્રોડક્ટ્સ પ્રાઇવેટ લિમિટેડ
KANNA

In [ ]:
import unicodedata
from collections import Counter
def contains_non_latin(text):
    if pd.isna(text):
        return False
    for ch in str(text):
        try:
            name = unicodedata.name(ch)
        except ValueError:
            continue
        if "LATIN" not in name and any(
            script in name
            for script in [
                "DEVANAGARI",
                "TELUGU",
                "KANNADA",
                "TAMIL",
                "GUJARATI",
                "BENGALI",
                "MALAYALAM",
                "ORIYA",
                "GURMUKHI"
            ]):
            return True
    return False

In [124]:
import unicodedata

def contains_non_latin(text):
    if pd.isna(text):
        return False

    for ch in str(text):
        try:
            char_name = unicodedata.name(ch)
        except ValueError:
            continue

        if "LATIN" not in char_name and any(
            script in char_name
            for script in [
                "DEVANAGARI",
                "TELUGU",
                "KANNADA",
                "TAMIL",
                "GUJARATI",
                "BENGALI",
                "MALAYALAM",
                "ORIYA",
                "GURMUKHI"
            ]
        ):
            return True

    return False

In [125]:
missed_non_latin = set()
missed_latin = set()
for chunk in pd.read_csv(
    RAW_S2_PATH,
    sep="\t",
    usecols=["entity_id", "business_name"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "business_name": "string"
    }
):
    for entity_id, name in zip(
        chunk["entity_id"],
        chunk["business_name"]
    ):
        if entity_id not in missed_s2_ids:
            continue
        if contains_non_latin(name):
            missed_non_latin.add(entity_id)
        else:
            missed_latin.add(entity_id)
    del chunk
    gc.collect()
print("Remaining missed S2 matches:", len(missed_s2_ids))
print("Missed with non-Latin names:", len(missed_non_latin))
print("Missed with Latin names:", len(missed_latin))

print(
    "Non-Latin percentage:",
    round(
        100 * len(missed_non_latin) / len(missed_s2_ids),
        4
    ),
    "%"
)

Remaining missed S2 matches: 471214
Missed with non-Latin names: 122682
Missed with Latin names: 348532
Non-Latin percentage: 26.0353 %


In [126]:
import subprocess
import sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "index", "versions", "indic-transliteration"],
    capture_output=True,
    text=True
)
print(result.stdout[:2000])

indic-transliteration (2.3.82)
Available versions: 2.3.82, 2.3.81, 2.3.79, 2.3.78, 2.3.76, 2.3.75, 2.3.74, 2.3.73, 2.3.72, 2.3.71, 2.3.70, 2.3.69, 2.3.68, 2.3.67, 2.3.64, 2.3.61, 2.3.60, 2.3.59, 2.3.57, 2.3.56, 2.3.55, 2.3.54, 2.3.53, 2.3.52, 2.3.51, 2.3.50, 2.3.49, 2.3.48, 2.3.47, 2.3.46, 2.3.45, 2.3.44, 2.3.43, 2.3.42, 2.3.41, 2.3.40, 2.3.39, 2.3.38, 2.3.37, 2.3.36, 2.3.35, 2.3.34, 2.3.33, 2.3.32, 2.3.31, 2.3.30, 2.3.29, 2.3.28, 2.3.27, 2.3.26, 2.3.25, 2.3.24, 2.3.22, 2.3.21, 2.3.20, 2.3.19, 2.3.18, 2.3.17, 2.3.13, 2.3.12, 2.3.11, 2.3.10, 2.3.9, 2.3.7, 2.3.5, 2.3.4, 2.3.3, 2.3.2, 2.3.0, 2.2.9, 2.2.8, 2.2.7, 2.2.6, 2.2.5, 2.2.4, 2.2.3, 2.2.2, 2.2.1, 2.2.0, 2.1.8, 2.1.7, 2.1.6, 2.1.5, 2.1.4, 2.1.3, 2.1.1, 2.1.0, 2.0.8, 2.0.7, 2.0.6, 2.0.5, 2.0.4, 2.0.3, 2.0.2, 2.0.1, 2.0.0, 1.9.9, 1.9.8, 1.9.7, 1.9.6, 1.9.5, 1.9.4, 1.9.3, 1.9.2, 1.9.1, 1.9.0, 1.8.9, 1.8.8, 1.8.7, 1.8.6, 1.8.5, 1.8.4, 1.8.3, 1.8.2, 1.8.1, 1.8.0, 1.7.9, 1.7.8, 1.7.7, 1.7.6, 1.7.5, 1.7.4, 1.7.3, 1.7.2, 1.7.1, 1.7.0, 1.6.9

In [127]:
!pip install indic-transliteration==2.3.82 -q

In [128]:
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
print("indic-transliteration imported successfully")

indic-transliteration imported successfully


In [129]:
test_names = [
    "राम मार्केटिंग प्राइवेट लिमिटेड",
    "குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்",
    "શક્તિ અર્બન પ્રોડક્ટ્સ પ્રાઇવેટ લિમિટેડ",
    "ಮಾಡರ್ನ್ ಕನ್ಸಲ್ಟೆಂಟ್ಸ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್",
    "গোল্ড প্রডিউসার স্টোর্স লিমিটেড",
    "శక్తి ఇంపెక్స్ ప్రైవేట్ లిమిటెడ్",
    "ശക്തി ഇംപെക്സ് പ്രൈവറ്റ് ലിമിറ്റഡ്",
    "ଭିଜନ୍ ଟେକ୍ନୋଲୋଜିସ୍ ପ୍ରାଇଭେଟ୍ ଲିମିଟେଡ୍",
    "ਸਕਾਈ ਈਸਟ ਇਸਟੇਟ ਲਿਮਿਟਡ"
]
script_pairs = [
    ("Devanagari", sanscript.DEVANAGARI),
    ("Tamil", sanscript.TAMIL),
    ("Gujarati", sanscript.GUJARATI),
    ("Kannada", sanscript.KANNADA),
    ("Bengali", sanscript.BENGALI),
    ("Telugu", sanscript.TELUGU),
    ("Malayalam", sanscript.MALAYALAM),
    ("Oriya", sanscript.ORIYA),
    ("Gurmukhi", sanscript.GURMUKHI),
]
for name in test_names:
    print("\nOriginal:", name)
    for script_name, source_script in script_pairs:
        try:
            result = transliterate(
                name,
                source_script,
                sanscript.ITRANS)
            print(f"{script_name:12s}: {result}")
        except Exception as e:
            print(f"{script_name:12s}: ERROR -> {e}")


Original: राम मार्केटिंग प्राइवेट लिमिटेड
Devanagari  : rAma mArkeTiMga prAiveTa limiTeDa
Tamil       : राम मार्केटिंग प्राइवेट लिमिटेड
Gujarati    : राम मार्केटिंग प्राइवेट लिमिटेड
Kannada     : राम मार्केटिंग प्राइवेट लिमिटेड
Bengali     : राम मार्केटिंग प्राइवेट लिमिटेड
Telugu      : राम मार्केटिंग प्राइवेट लिमिटेड
Malayalam   : राम मार्केटिंग प्राइवेट लिमिटेड
Oriya       : राम मार्केटिंग प्राइवेट लिमिटेड
Gurmukhi    : राम मार्केटिंग प्राइवेट लिमिटेड

Original: குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்
Devanagari  : குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்
Tamil       : ghuLobhal bhijhiனs bhiraiveDh limiDhèDh
Gujarati    : குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்
Kannada     : குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்
Bengali     : குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்
Telugu      : குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்
Malayalam   : குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்
Oriya       : குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்
Gurmukhi    : குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்

Original: શક્તિ અર્બન પ્રોડક્ટ્સ પ્રાઇવેટ લિમિટેડ
Devanag

In [130]:
import unicodedata
from indic_transliteration.sanscript import transliterate, DEVANAGARI, IAST
test_cases = [
    ("Devanagari", "राम मार्केटिंग प्राइवेट लिमिटेड", DEVANAGARI),
    ("Tamil", "குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்", sanscript.TAMIL),
    ("Gujarati", "શક્તિ અર્બન પ્રોડક્ટ્સ પ્રાઇવેટ લિમિટેડ", sanscript.GUJARATI),
    ("Kannada", "ಮಾಡರ್ನ್ ಕನ್ಸಲ್ಟೆಂಟ್ಸ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್", sanscript.KANNADA),
    ("Bengali", "গোল্ড প্রডিউসার স্টোর্স লিমিটেড", sanscript.BENGALI),
    ("Telugu", "శక్తి ఇంపెక్స్ ప్రైవేట్ లిమిటెడ్", sanscript.TELUGU),
    ("Malayalam", "ശക്തി ഇംപെക്സ് പ്രൈവറ്റ് ലിമിറ്റഡ്", sanscript.MALAYALAM),
    ("Oriya", "ଭିଜନ୍ ଟେକ୍ନୋଲୋଜିସ୍ ପ୍ରାଇଭେଟ୍ ଲିମିଟେଡ୍", sanscript.ORIYA),
    ("Gurmukhi", "ਸਕਾਈ ਈਸਟ ਇਸਟੇਟ ਲਿਮਿਟਡ", sanscript.GURMUKHI),
]

def strip_diacritics(text):
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(ch)
    )
for script_name, text, source_script in test_cases:
    result = transliterate(text, source_script, IAST)
    normalized = strip_diacritics(result).lower()
    print(f"\n{script_name}")
    print("Original :", text)
    print("IAST     :", result)
    print("Normalized:", normalized)


Devanagari
Original : राम मार्केटिंग प्राइवेट लिमिटेड
IAST     : rāma mārkeṭiṃga prāiveṭa limiṭeḍa
Normalized: rama marketimga praiveta limiteda

Tamil
Original : குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்
IAST     : ghuḻobhal bhijhiṉas bhiraiveḍh limiḍhèḍh
Normalized: ghulobhal bhijhinas bhiraivedh limidhedh

Gujarati
Original : શક્તિ અર્બન પ્રોડક્ટ્સ પ્રાઇવેટ લિમિટેડ
IAST     : śakti arbana proḍakṭsa prāiveṭa limiṭeḍa
Normalized: sakti arbana prodaktsa praiveta limiteda

Kannada
Original : ಮಾಡರ್ನ್ ಕನ್ಸಲ್ಟೆಂಟ್ಸ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್
IAST     : māḍarn kansalṭèṃṭs praiveṭ limiṭèḍ
Normalized: madarn kansaltemts praivet limited

Bengali
Original : গোল্ড প্রডিউসার স্টোর্স লিমিটেড
IAST     : golḍa praḍiusāra sṭorsa limiṭeḍa
Normalized: golda pradiusara storsa limiteda

Telugu
Original : శక్తి ఇంపెక్స్ ప్రైవేట్ లిమిటెడ్
IAST     : śakti iṃpèks praiveṭ limiṭèḍ
Normalized: sakti impeks praivet limited

Malayalam
Original : ശക്തി ഇംപെക്സ് പ്രൈവറ്റ് ലിമിറ്റഡ്
IAST     : śakti iṃpèks praivaṟṟ limiṟṟaḍ
Norma

In [131]:
NAME_STOPWORDS = {
    "inc", "incorporated", "llc", "ltd", "limited",
    "pvt", "private", "company", "co", "corp",
    "corporation", "group", "services", "partners",
    "holdings", "associates", "center", "india",
    "and", "of"
}

def meaningful_tokens(name):
    if pd.isna(name) or not str(name).strip():
        return []

    tokens = str(name).split()

    return [
        token
        for token in tokens
        if token not in NAME_STOPWORDS and len(token) >= 2
    ]

print("meaningful_tokens restored")

meaningful_tokens restored


In [132]:
missed_s2_ids = set()

total_true = 0
captured_existing = 0

for row in s1_blocking.itertuples(index=False):

    s1_id = row.entity_id
    country = row.country_norm
    name = row.name_core
    address = row.address_norm

    true_ids = {
        x
        for x in gt_lookup.get(s1_id, set())
        if x.startswith("S2-")
    }

    if not true_ids:
        continue

    total_true += len(true_ids)

    candidate_set = set()

    # --------------------------------------------
    # 1. Exact name blocker
    # --------------------------------------------
    if pd.notna(name) and name != "":
        candidate_set.update(
            s2_name_index.get((country, name), [])
        )

        # ----------------------------------------
        # 2. Name-token blocker
        # ----------------------------------------
        for token in set(meaningful_tokens(name)):
            if token in allowed_name_tokens:
                candidate_set.update(
                    name_token_index.get(token, [])
                )

    # --------------------------------------------
    # 3. Address-token blocker
    # --------------------------------------------
    if pd.notna(address) and address != "":
        for token in set(str(address).split()):
            if token in allowed_address_tokens:
                candidate_set.update(
                    address_token_index.get(token, [])
                )

    captured = true_ids.intersection(candidate_set)

    captured_existing += len(captured)

    missed_s2_ids.update(
        true_ids - captured
    )

print("Total true S2 matches:", total_true)
print("Captured by existing blocker:", captured_existing)
print("Missed by existing blocker:", len(missed_s2_ids))

Total true S2 matches: 3693619
Captured by existing blocker: 3222405
Missed by existing blocker: 471214


In [133]:
missed_non_latin = set()
missed_latin = set()

for chunk in pd.read_csv(
    RAW_S2_PATH,
    sep="\t",
    usecols=["entity_id", "business_name"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "business_name": "string"
    }
):
    for entity_id, name in zip(
        chunk["entity_id"],
        chunk["business_name"]
    ):
        if entity_id not in missed_s2_ids:
            continue

        if contains_non_latin(name):
            missed_non_latin.add(entity_id)
        else:
            missed_latin.add(entity_id)

    del chunk
    gc.collect()

print("Remaining missed S2 matches:", len(missed_s2_ids))
print("Missed with non-Latin names:", len(missed_non_latin))
print("Missed with Latin names:", len(missed_latin))

print(
    "Non-Latin percentage:",
    round(
        100 * len(missed_non_latin) / len(missed_s2_ids),
        4
    ),
    "%"
)

Remaining missed S2 matches: 471214
Missed with non-Latin names: 122682
Missed with Latin names: 348532
Non-Latin percentage: 26.0353 %


In [134]:
# Collect only the S2 records that are:
# 1. genuine ground-truth matches
# 2. currently missed by our blocker
# 3. have a non-Latin business name

missed_non_latin_records = []

for chunk in pd.read_csv(
    RAW_S2_PATH,
    sep="\t",
    usecols=["entity_id", "business_name"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "business_name": "string"
    }
):
    mask = chunk["entity_id"].isin(missed_non_latin)

    selected = chunk.loc[mask]

    if len(selected) > 0:
        missed_non_latin_records.append(selected)

    del chunk
    gc.collect()

missed_non_latin_df = pd.concat(
    missed_non_latin_records,
    ignore_index=True
)

del missed_non_latin_records
gc.collect()

print("Records collected:", len(missed_non_latin_df))
print(missed_non_latin_df.head(10))

Records collected: 122682
      entity_id                                 business_name
0  S2-566688803                       रियल मॉडर्न फूड लिमिटेड
1  S2-879667609             குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்
2   S2-64510343         अल्फा अल टेक्नोलॉजीज प्राइवेट लिमिटेड
3  S2-631299191                 ग्लोबल इन्वेस्टमेंट प्रा. लि.
4  S2-906753214                   फर्स्ट फूड प्राइवेट लिमिटेड
5  S2-923049280              પ્રાઇમ બાલાજી ટ્રેડિંગ પ્રા. લિ.
6  S2-938256871       ডায়নামিক ইন্ডাস্ট্রিজ প্রাইভেট লিমিটেড
7  S2-293759502  పర్‌ఫెక్ట్ యునైటెడ్ మీడియా ప్రైవేట్ లిమిటెడ్
8  S2-625988543      సన్ బెస్ట్ సొల్యూషన్స్ ప్రైవేట్ లిమిటెడ్
9  S2-719478517                                  आनंद वेंचर्स


In [135]:
missed_pairs_for_translit = []
for row in s1_blocking.itertuples(index=False):
    s1_id = row.entity_id
    true_s2_ids = {
        x for x in gt_lookup.get(s1_id, set())
        if x.startswith("S2-") and x in missed_non_latin
    }
    if not true_s2_ids:
        continue

    for s2_id in true_s2_ids:
        missed_pairs_for_translit.append({
            "s1_id": s1_id,
            "s1_name": row.name_core,
            "s2_id": s2_id
        })

missed_pairs_for_translit = pd.DataFrame(
    missed_pairs_for_translit
)

print("Missed non-Latin true pairs:", len(missed_pairs_for_translit))
print(missed_pairs_for_translit.head())

Missed non-Latin true pairs: 122682
          s1_id          s1_name         s2_id
0  S1-597762257  green logistics  S2-152865025
1  S1-597762257  green logistics  S2-750318376
2  S1-597762257  green logistics  S2-535895394
3  S1-666876590       ss systems  S2-975207252
4  S1-666876590       ss systems  S2-185860935


In [136]:
missed_pairs_for_translit = missed_pairs_for_translit.merge(
    missed_non_latin_df,
    left_on="s2_id",
    right_on="entity_id",
    how="left"
)
missed_pairs_for_translit = missed_pairs_for_translit.rename(
    columns={"business_name": "s2_name"}
)
print(
    missed_pairs_for_translit[
        ["s1_id", "s1_name", "s2_id", "s2_name"]
    ].head(20).to_string(index=False)
)

       s1_id              s1_name        s2_id                                s2_name
S1-597762257      green logistics S2-152865025     ग्रीन लॉजिस्टिक्स प्राइवेट लिमिटेड
S1-597762257      green logistics S2-750318376     ग्रीन लॉजिस्टिक्स प्राइवेट लिमिटेड
S1-597762257      green logistics S2-535895394     ग्रीन लॉजिस्टिक्स प्राइवेट लिमिटेड
S1-666876590           ss systems S2-975207252                  एसएस सिस्टम्स लिमिटेड
S1-666876590           ss systems S2-185860935                  एसएस सिस्टम्स लिमिटेड
S1-366329737 silver constructions  S2-23560385             सिल्वर कंस्ट्रक्शंस एलएलपी
S1-912508322    shiv technologies S2-276782865             ಶಿವ್ ಟೆಕ್ನಾಲಜೀಸ್ ಎಲ್ಎಲ್‌ಪಿ
S1-216511119         surya energy S2-645205818          सूर्य एनर्जी प्राइवेट लिमिटेड
S1-499302167     shree industries  S2-45522817      શ્રી ઇન્ડસ્ટ્રીઝ પ્રાઇવેટ લિમિટેડ
S1-499302167     shree industries S2-127669900      શ્રી ઇન્ડસ્ટ્રીઝ પ્રાઇવેટ લિમિટેડ
S1-499302167     shree industries S2-148775637      શ્

In [137]:
from indic_transliteration.sanscript import transliterate, IAST
from indic_transliteration import sanscript
import unicodedata

In [138]:
SCRIPT_MAP = {
    "DEVANAGARI": sanscript.DEVANAGARI,
    "TELUGU": sanscript.TELUGU,
    "KANNADA": sanscript.KANNADA,
    "TAMIL": sanscript.TAMIL,
    "GUJARATI": sanscript.GUJARATI,
    "BENGALI": sanscript.BENGALI,
    "MALAYALAM": sanscript.MALAYALAM,
    "ORIYA": sanscript.ORIYA,
    "GURMUKHI": sanscript.GURMUKHI,
}

In [139]:
def normalize_transliteration(text):
    if pd.isna(text):
        return ""

    text = str(text)
    scripts_found = set()

    for ch in text:
        try:
            char_name = unicodedata.name(ch)
        except ValueError:
            continue

        for script_name in SCRIPT_MAP:
            if script_name in char_name:
                scripts_found.add(script_name)
    if not scripts_found:
        return text.lower()

    result = text
    for script_name in scripts_found:
        source_script = SCRIPT_MAP[script_name]

        try:
            result = transliterate(
                result,
                source_script,
                IAST
            )
        except Exception:
            pass
    result = "".join(
        ch for ch in unicodedata.normalize("NFKD", result)
        if not unicodedata.combining(ch)
    )
    result = result.lower()
    result = " ".join(result.split())

    return result

In [140]:
sample_translit = missed_pairs_for_translit.head(1000).copy()
sample_translit["s2_translit"] = (
    sample_translit["s2_name"]
    .apply(normalize_transliteration)
)
print(
    sample_translit[
        ["s1_name", "s2_name", "s2_translit"]
    ].head(30).to_string(index=False)
)

              s1_name                                   s2_name                                 s2_translit
      green logistics        ग्रीन लॉजिस्टिक्स प्राइवेट लिमिटेड         grina laॉjistiksa praiveta limiteda
      green logistics        ग्रीन लॉजिस्टिक्स प्राइवेट लिमिटेड         grina laॉjistiksa praiveta limiteda
      green logistics        ग्रीन लॉजिस्टिक्स प्राइवेट लिमिटेड         grina laॉjistiksa praiveta limiteda
           ss systems                     एसएस सिस्टम्स लिमिटेड                    esaesa sistamsa limiteda
           ss systems                     एसएस सिस्टम्स लिमिटेड                    esaesa sistamsa limiteda
 silver constructions                सिल्वर कंस्ट्रक्शंस एलएलपी              silvara kamstraksamsa elaelapi
    shiv technologies                ಶಿವ್ ಟೆಕ್ನಾಲಜೀಸ್ ಎಲ್ಎಲ್‌ಪಿ                      siv teknalajis elel‌pi
         surya energy             सूर्य एनर्जी प्राइवेट लिमिटेड              surya enarji praiveta limiteda
     shree industries       

In [141]:
from difflib import SequenceMatcher
def char_similarity(a, b):
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, str(a), str(b)).ratio()

def ngrams(text, n=3):
    text = str(text).replace(" ", "")
    if len(text) < n:
        return {text} if text else set()
    return {
        text[i:i+n]
        for i in range(len(text) - n + 1)
    }

def jaccard_similarity(a, b):
    if not a or not b:
        return 0.0
    a_set = set(a)
    b_set = set(b)
    if not a_set or not b_set:
        return 0.0
    return len(a_set & b_set) / len(a_set | b_set)

In [142]:
sample_translit = missed_pairs_for_translit.copy()

sample_translit["s2_translit"] = (
    sample_translit["s2_name"]
    .apply(normalize_transliteration)
)
sample_translit["translit_char_sim"] = sample_translit.apply(
    lambda row: char_similarity(
        row["s1_name"],
        row["s2_translit"]
    ),
    axis=1
)
sample_translit["translit_3gram_jaccard"] = sample_translit.apply(
    lambda row: jaccard_similarity(
        ngrams(row["s1_name"], 3),
        ngrams(row["s2_translit"], 3)
    ),
    axis=1
)
sample_translit["translit_token_jaccard"] = sample_translit.apply(
    lambda row: jaccard_similarity(
        str(row["s1_name"]).split(),
        str(row["s2_translit"]).split()
    ),
    axis=1
)
print(
    sample_translit[
        [
            "s1_name",
            "s2_name",
            "s2_translit",
            "translit_char_sim",
            "translit_3gram_jaccard",
            "translit_token_jaccard"
        ]
    ].head(20).to_string(index=False)
)

             s1_name                                s2_name                         s2_translit  translit_char_sim  translit_3gram_jaccard  translit_token_jaccard
     green logistics     ग्रीन लॉजिस्टिक्स प्राइवेट लिमिटेड grina laॉjistiksa praiveta limiteda           0.400000                0.050000                     0.0
     green logistics     ग्रीन लॉजिस्टिक्स प्राइवेट लिमिटेड grina laॉjistiksa praiveta limiteda           0.400000                0.050000                     0.0
     green logistics     ग्रीन लॉजिस्टिक्स प्राइवेट लिमिटेड grina laॉjistiksa praiveta limiteda           0.400000                0.050000                     0.0
          ss systems                  एसएस सिस्टम्स लिमिटेड            esaesa sistamsa limiteda           0.470588                0.000000                     0.0
          ss systems                  एसएस सिस्टम्स लिमिटेड            esaesa sistamsa limiteda           0.470588                0.000000                     0.0
silver constructions  

In [143]:
for col in [
    "translit_char_sim",
    "translit_3gram_jaccard",
    "translit_token_jaccard"
]:
    print("\n", "=" * 60)
    print(col)

    print(
        sample_translit[col].describe(
            percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]
        ))
    for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        recovered = (
            sample_translit[col] >= threshold
        ).sum()
        print(
            f">= {threshold:.1f}: "
            f"{recovered:,} "
            f"({recovered / len(sample_translit) * 100:.2f}%)"
        )


translit_char_sim
count    122682.000000
mean          0.465579
std           0.121316
min           0.072727
25%           0.384615
50%           0.457143
75%           0.533333
90%           0.619048
95%           0.682927
max           1.000000
Name: translit_char_sim, dtype: float64
>= 0.3: 114,748 (93.53%)
>= 0.4: 87,646 (71.44%)
>= 0.5: 45,028 (36.70%)
>= 0.6: 15,665 (12.77%)
>= 0.7: 5,146 (4.19%)
>= 0.8: 1,694 (1.38%)
>= 0.9: 260 (0.21%)

translit_3gram_jaccard
count    122682.000000
mean          0.090578
std           0.092405
min           0.000000
25%           0.027027
50%           0.066667
75%           0.127660
90%           0.200000
95%           0.263158
max           1.000000
Name: translit_3gram_jaccard, dtype: float64
>= 0.3: 4,297 (3.50%)
>= 0.4: 1,678 (1.37%)
>= 0.5: 745 (0.61%)
>= 0.6: 298 (0.24%)
>= 0.7: 110 (0.09%)
>= 0.8: 65 (0.05%)
>= 0.9: 47 (0.04%)

translit_token_jaccard
count    122682.000000
mean          0.037375
std           0.091953
min           0.

In [144]:
post_3gram_non_latin = missed_non_latin.intersection(missed_s2_ids)
print("Non-Latin baseline misses:", len(missed_non_latin))
print("Remaining non-Latin misses after 3-gram:", len(post_3gram_non_latin))

Non-Latin baseline misses: 122682
Remaining non-Latin misses after 3-gram: 122682
